### Conditional Chains

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser,StrOutputParser
from pydantic import BaseModel,Field
from typing import Literal

In [3]:
class feedback(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description='the sentiment of the review.')

In [4]:
parser = PydanticOutputParser(pydantic_object=feedback)

In [5]:
model = ChatOpenAI(model ='gpt-4o-mini')

In [6]:
prompt1 = PromptTemplate(
    template='what is the sentiment of the review\n{review}\n{format_instructions}',
    input_variables = ['review'],
    partial_variables={'format_instructions':parser.get_format_instructions()}
)

### how the prompt will look like after the format_instructions()

In [7]:
prompt1.invoke(input ={'review':'this is a terrible phone'})

StringPromptValue(text='what is the sentiment of the review\nthis is a terrible phone\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"sentiment": {"description": "the sentiment of the review.", "enum": ["positive", "negative"], "title": "Sentiment", "type": "string"}}, "required": ["sentiment"]}\n```')

In [8]:
sentiment_chain = prompt1 | model | parser 

In [9]:
sentiment_chain.invoke(input={'review':'this is a beautiful phone'})

feedback(sentiment='positive')

In [14]:
prompt2 = PromptTemplate(
    template = 'write one reply for this positive sentiment review {review}',
    input_variables=['review']
)

In [15]:
prompt3 = PromptTemplate(
    template = 'write one reply for this negative sentiment review {review}',
    input_variables=['review']
)

In [13]:
parser2 = StrOutputParser()

In [22]:
from langchain_core.runnables import RunnableBranch,RunnableLambda

In [25]:
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive',prompt2 | model | parser2),
    (lambda x: x.sentiment =='negative',prompt3 | model | parser2),
    RunnableLambda(lambda x:'could not find sentiment')
)

In [26]:
main_chain = sentiment_chain | branch_chain

In [27]:
main_chain.invoke({'review':'the smartphone is rubbish'})

"Of course! Here's a response you could use for a negative sentiment review:\n\n---\n\nDear [Reviewer’s Name],\n\nThank you for taking the time to share your thoughts with us. We genuinely regret to hear that your experience did not meet your expectations. Your feedback is invaluable, and we take it seriously as it helps us improve our services.\n\nWe would love the opportunity to make this right. Please feel free to reach out to us directly at [contact information] so we can address your concerns personally.\n\nThank you for giving us the chance to improve.\n\nBest regards,  \n[Your Name]  \n[Your Position]  \n[Your Company]  \n\n--- \n\nFeel free to customize the response as needed!"

In [28]:
main_chain.invoke({'review':'its overall budgetfriendly phone'})

"Thank you so much for your kind words! We're thrilled to hear that you had a positive experience. Your support means a lot to us, and we look forward to serving you again soon!"